# Pandas для Data Engineering

Комплексний урок з маніпуляції даними за допомогою Pandas та знайомство з Polars.

---
**Рівень:** Junior–Middle Data Engineer  
**Тривалість:** ~4 години (теорія + практика)  
**Що вивчите:** читання/запис, очищення, трансформація, агрегація, об'єднання, робота з часом, Polars

In [18]:
# import libraries
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

print(f'Pandas version: {pd.__version__}')

Pandas version: 3.0.3


---
## 1. Вступ: чому Pandas у Data Engineering?

**Pandas** — основна бібліотека Python для роботи з табличними даними. У Data Engineering ми щодня:
- читаємо дані з файлів (CSV, Excel, Parquet, JSON) і баз даних (SQL)
- чистимо та трансформуємо дані (null, дублікати, типи)
- робимо агрегації та зведення (groupby, pivot)
- об'єднуємо джерела (merge, join, concat)
- готуємо дані для завантаження в DWH / Data Lake

> **DE-підхід:** пишемо ETL-пайплайни, де Pandas — двигун трансформації між Extract та Load.

---
## 2. Основні структури: Series та DataFrame

- **Series** — одновимірний масив з індексом (стовпець)
- **DataFrame** — двовимірна таблиця (колекція Series)

In [19]:
# Series
s = pd.Series([10, 20, 30, 40, 50], name='score')
s

0    10
1    20
2    30
3    40
4    50
Name: score, dtype: int64

In [20]:
# DataFrame зі словника
df = pd.DataFrame({
    'user_id': [1, 2, 3, 4, 5],
    'name': ['Alice', 'Bob', 'Charlie', 'Diana', 'Eve'],
    'age': [25, 30, 35, 28, 40],
    'salary': [50000, 60000, 70000, 55000, 80000],
    'department': ['IT', 'HR', 'IT', 'Finance', 'HR']
})
df

,user_id,name,age,salary,department
0,1,Alice,25,50000,IT
1,2,Bob,30,60000,HR
2,3,Charlie,35,70000,IT
3,4,Diana,28,55000,Finance
4,5,Eve,40,80000,HR


In [21]:
# Корисні атрибути
print('Shape:', df.shape)
print('Columns:', df.columns.tolist())
print('Index:', df.index)
print('Dtypes:\n', df.dtypes)
print('\nValues (numpy array):', df.values.shape)

Shape: (5, 5)
Columns: ['user_id', 'name', 'age', 'salary', 'department']
Index: RangeIndex(start=0, stop=5, step=1)
Dtypes:
 user_id       int64
name            str
age           int64
salary        int64
department      str
dtype: object

Values (numpy array): (5, 5)


In [22]:
# Індексація
print('Перші 2 рядки:')
display(df.head(2))

print('Доступ до стовпця:', df['name'].tolist())
print('Доступ до рядка за індексом (iloc):', df.iloc[2].to_dict())
print('Доступ за міткою (loc):', df.loc[df['name'] == 'Diana', ['name', 'salary']].to_dict('records'))

Перші 2 рядки:


,user_id,name,age,salary,department
0,1,Alice,25,50000,IT
1,2,Bob,30,60000,HR


Доступ до стовпця: ['Alice', 'Bob', 'Charlie', 'Diana', 'Eve']
Доступ до рядка за індексом (iloc): {'user_id': 3, 'name': 'Charlie', 'age': 35, 'salary': 70000, 'department': 'IT'}
Доступ за міткою (loc): [{'name': 'Diana', 'salary': 55000}]


---
## 3. Читання та запис даних (I/O)

Для DE критично вміти працювати з різними форматами.

In [23]:
# Створимо демо-дані для експорту
import os
demo_dir = 'demo_data'
os.makedirs(demo_dir, exist_ok=True)

# ---- CSV ----
df.to_csv(f'{demo_dir}/employees.csv', index=False)
df_csv = pd.read_csv(f'{demo_dir}/employees.csv')
print('CSV OK:', df_csv.shape)

# ---- Excel ----
df.to_excel(f'{demo_dir}/employees.xlsx', index=False, sheet_name='employees')
df_xl = pd.read_excel(f'{demo_dir}/employees.xlsx', sheet_name='employees')
print('Excel OK:', df_xl.shape)

# ---- JSON ----
df.to_json(f'{demo_dir}/employees.json', orient='records', lines=True)
df_json = pd.read_json(f'{demo_dir}/employees.json', lines=True)
print('JSON (lines) OK:', df_json.shape)

# ---- Parquet ----
try:
    df.to_parquet(f'{demo_dir}/employees.parquet', index=False)
    df_pq = pd.read_parquet(f'{demo_dir}/employees.parquet')
    print('Parquet OK:', df_pq.shape)
except Exception as e:
    print(f'Parquet not available (install pyarrow): {e}')

CSV OK: (5, 5)
Excel OK: (5, 5)
JSON (lines) OK: (5, 5)
Parquet OK: (5, 5)


In [24]:
# ---- SQL (SQLite in-memory) ----
from sqlalchemy import create_engine

engine = create_engine('sqlite:///:memory:')
df.to_sql('employees', engine, index=False, if_exists='replace')

# Read SQL query
df_sql = pd.read_sql('SELECT * FROM employees WHERE salary > 55000', engine)
print('SQL query result:')
display(df_sql)

# Read entire table
df_sql_full = pd.read_sql_table('employees', engine)
print('Full table from SQL:', df_sql_full.shape)

SQL query result:


,user_id,name,age,salary,department
0,2,Bob,30,60000,HR
1,3,Charlie,35,70000,IT
2,5,Eve,40,80000,HR


Full table from SQL: (5, 5)


In [25]:
# ---- Парсинг великих CSV (chunking) ----
# Корисно для файлів, які не влазять у RAM
# Створимо великий файл для демонстрації
big_df = pd.DataFrame({'id': range(100000), 'value': np.random.randn(100000)})
big_df.to_csv(f'{demo_dir}/big_file.csv', index=False)

# Читаємо chunks
chunk_iter = pd.read_csv(f'{demo_dir}/big_file.csv', chunksize=10000)
total = 0
for chunk in chunk_iter:
    total += chunk['value'].sum()
print(f'Сума значень через chunking: {total:.2f}')

# Рекомендація: використовуйте low_memory=False, dtype={'col': type} для економії пам'яті

Сума значень через chunking: -91.47


---
## 4. Розвідка даних (EDA)

Перед трансформацією — завжди досліджуйте дані!

In [26]:
# Створимо реалістичніший датасет
np.random.seed(42)
n = 1000

df_de = pd.DataFrame({
    'user_id': range(1, n + 1),
    'name': np.random.choice(['Alice', 'Bob', 'Charlie', 'Diana', 'Eve', None], size=n, p=[0.2, 0.2, 0.2, 0.15, 0.15, 0.1]),
    'age': np.random.randint(18, 65, size=n).astype(float),
    'salary': np.round(np.random.lognormal(mean=10.5, sigma=0.5, size=n), 0),
    'department': np.random.choice(['IT', 'HR', 'Finance', 'Marketing', 'Sales'], size=n),
    'join_date': [datetime(2018, 1, 1) + timedelta(days=int(i)) for i in np.random.randint(0, 1200, size=n)],
    'is_active': np.random.choice([True, False], size=n, p=[0.85, 0.15]),
    'score': np.random.uniform(0, 100, size=n).round(1)
})

# Додамо трохи пропусків
df_de.loc[::50, 'salary'] = np.nan
df_de.loc[::30, 'score'] = np.nan
df_de.loc[::70, 'department'] = np.nan

print('Реалістичний датасет:', df_de.shape)
display(df_de.head())

Реалістичний датасет: (1000, 8)


,user_id,name,age,salary,department,join_date,is_active,score
0,1,Bob,64.0,NaN,NaN,2019-11-01,True,NaN
1,2,NaN,29.0,55229.0,HR,2018-05-29,True,84.0
2,3,Diana,33.0,37832.0,HR,2020-09-11,True,1.7
3,4,Charlie,41.0,34564.0,Finance,2020-05-01,True,4.9
4,5,Alice,36.0,57500.0,HR,2019-09-20,False,57.3


In [27]:
# Базова розвідка
print('=== HEAD ===')
display(df_de.head(3))

print('\n=== INFO ===')
df_de.info(show_counts=True)

print('\n=== DESCRIBE (числові) ===')
display(df_de.describe())

print('\n=== DESCRIBE (категоріальні) ===')
display(df_de.describe(include=['object', 'bool']))

=== HEAD ===


,user_id,name,age,salary,department,join_date,is_active,score
0,1,Bob,64.0,NaN,NaN,2019-11-01,True,NaN
1,2,NaN,29.0,55229.0,HR,2018-05-29,True,84.0
2,3,Diana,33.0,37832.0,HR,2020-09-11,True,1.7



=== INFO ===
<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   user_id     1000 non-null   int64         
 1   name        900 non-null    str           
 2   age         1000 non-null   float64       
 3   salary      980 non-null    float64       
 4   department  985 non-null    str           
 5   join_date   1000 non-null   datetime64[us]
 6   is_active   1000 non-null   bool          
 7   score       966 non-null    float64       
dtypes: bool(1), datetime64[us](1), float64(3), int64(1), str(2)
memory usage: 65.0 KB

=== DESCRIBE (числові) ===


,user_id,age,salary,join_date,score
count,1000.000000,1000.000000,980.000000,1000,966.000000
mean,500.500000,40.984000,41328.626531,2019-08-20 13:32:09.600000,49.450828
min,1.000000,18.000000,8024.000000,2018-01-01 00:00:00,0.400000
25%,250.750000,29.000000,26586.500000,2018-10-21 00:00:00,23.825000
50%,500.500000,41.000000,36331.000000,2019-08-22 00:00:00,49.600000
75%,750.250000,53.000000,50977.250000,2020-06-12 00:00:00,74.975000
max,1000.000000,64.000000,179253.000000,2021-04-13 00:00:00,99.900000
std,288.819436,13.582939,21363.792887,NaN,29.160445



=== DESCRIBE (категоріальні) ===


,name,department,is_active
count,900,985,1000
unique,5,5,2
top,Alice,Finance,True
freq,225,210,848


In [28]:
# Аналіз пропусків
missing = df_de.isnull().sum()
missing_pct = (df_de.isnull().sum() / len(df_de) * 100).round(2)

missing_report = pd.DataFrame({'missing_count': missing, 'missing_%': missing_pct})
print('=== Пропуски в даних ===')
display(missing_report[missing_report['missing_count'] > 0].sort_values('missing_count', ascending=False))

=== Пропуски в даних ===


,missing_count,missing_%
name,100,10.0
score,34,3.4
salary,20,2.0
department,15,1.5


In [29]:
# Унікальні значення
for col in ['department', 'is_active', 'name']:
    print(f'{col}: {df_de[col].nunique()} унікальних значень')
    if df_de[col].dtype == 'object' or df_de[col].dtype == 'bool':
        display(df_de[col].value_counts(dropna=False).head(10))

department: 5 унікальних значень
is_active: 2 унікальних значень


is_active
True     848
False    152
Name: count, dtype: int64

name: 5 унікальних значень


In [30]:
# Дослідження дублікатів
print(f'Повних дублікатів: {df_de.duplicated().sum()}')
print(f'Дублікатів за user_id: {df_de["user_id"].duplicated().sum()}')
print(f'Дублікатів за user_id + department: {df_de.duplicated(subset=["user_id", "department"]).sum()}')

Повних дублікатів: 0
Дублікатів за user_id: 0
Дублікатів за user_id + department: 0


---
## 5. Очищення даних (Data Cleaning)

80% роботи DE — це очищення даних.

In [31]:
# 5.1 Пропуски: стратегії
df_clean = df_de.copy()

# Видалення рядків з пропусками
print('До dropna:', df_clean.shape)
print('Після dropna (any):', df_clean.dropna().shape)
print('Після dropna (всі стовпці):', df_clean.dropna(how='all').shape)
print('Після dropna (thresh=5):', df_clean.dropna(thresh=5).shape)
print('Після dropna (подмножина):', df_clean.dropna(subset=['salary', 'score']).shape)

До dropna: (1000, 8)
Після dropna (any): (853, 8)
Після dropna (всі стовпці): (1000, 8)
Після dropna (thresh=5): (1000, 8)
Після dropna (подмножина): (953, 8)


In [32]:
# Заповнення пропусків
df_filled = df_clean.copy()

# Середнім / медіаною (для числових)
df_filled['salary'] = df_filled['salary'].fillna(df_filled['salary'].median())
df_filled['score'] = df_filled['score'].fillna(df_filled['score'].mean())

# Константою / модою (для категоріальних)
df_filled['department'] = df_filled['department'].fillna('Unknown')
df_filled['name'] = df_filled['name'].fillna(df_filled['name'].mode()[0])

# Forward / Backward fill (для часових рядів)
# df_filled = df_filled.fillna(method='ffill')  # or 'bfill'

# Інтерполяція
# df_filled['score'] = df_filled['score'].interpolate(method='linear')

print('Пропусків після fillna:')
display(df_filled.isnull().sum().to_frame('missing'))

Пропусків після fillna:


,missing
user_id,0
name,0
age,0
salary,0
department,0
join_date,0
is_active,0
score,0


In [33]:
# 5.2 Дублікати
# Додамо дублікат для демонстрації
df_dup = pd.concat([df_clean, df_clean.iloc[[0]]], ignore_index=True)
print('З дублікатом:', df_dup.shape)
df_dup = df_dup.drop_duplicates()
print('Після drop_duplicates:', df_dup.shape)

# За вказаними колонками
# df_clean = df_clean.drop_duplicates(subset=['user_id'], keep='last')

З дублікатом: (1001, 8)
Після drop_duplicates: (1000, 8)


In [34]:
# 5.3 Типи даних (конвертація)
print('=== До конвертації ===')
print(df_clean.dtypes)

# astype
df_clean['user_id'] = df_clean['user_id'].astype(str)  # int -> str
df_clean['age'] = df_clean['age'].astype(int)  # float -> int
df_clean['is_active'] = df_clean['is_active'].astype(int)  # bool -> int (0/1)

# Категоріальний тип (економить пам'ять)
df_clean['department'] = df_clean['department'].astype('category')

# Дати
df_clean['join_date'] = pd.to_datetime(df_clean['join_date'])
# df_clean['join_date_ymd'] = df_clean['join_date'].dt.date

print('\n=== Після конвертації ===')
print(df_clean.dtypes)
print(f'\nMemory usage: {df_clean.memory_usage(deep=True).sum() / 1024:.1f} KB')

=== До конвертації ===
user_id                int64
name                     str
age                  float64
salary               float64
department               str
join_date     datetime64[us]
is_active               bool
score                float64
dtype: object

=== Після конвертації ===
user_id                  str
name                     str
age                    int32
salary               float64
department          category
join_date     datetime64[us]
is_active              int32
score                float64
dtype: object

Memory usage: 55.1 KB


In [35]:
# 5.4 Викиди (outliers) — метод IQR
Q1 = df_clean['salary'].quantile(0.25)
Q3 = df_clean['salary'].quantile(0.75)
IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

outliers_mask = (df_clean['salary'] < lower) | (df_clean['salary'] > upper)
print(f'Знайдено викидів salary: {outliers_mask.sum()}')
print(f'Межі: нижня={lower:.0f}, верхня={upper:.0f}')

# Варіанти обробки:
# 1. Видалити: df_clean = df_clean[~outliers_mask]
# 2. Обмежити (capping): df_clean['salary'] = df_clean['salary'].clip(lower, upper)
# 3. Залишити (якщо це легітимні значення)

Знайдено викидів salary: 34
Межі: нижня=-10000, верхня=87563


---
## 6. Фільтрація та сортування

Базові, але критичні операції для DE.

In [36]:
df_f = df_de.copy()

# Boolean indexing (найшвидший спосіб фільтрації)
mask_it = df_f['department'] == 'IT'
mask_salary = df_f['salary'] > 60000
mask_active = df_f['is_active'] == True

filtered = df_f[mask_it & mask_salary & mask_active]
print(f'Активні IT-співробітники з salary > 60000: {len(filtered)}')
display(filtered[['name', 'department', 'salary']].head())

Активні IT-співробітники з salary > 60000: 29


,name,department,salary
95,Charlie,IT,79281.0
106,Charlie,IT,66185.0
110,Bob,IT,65700.0
145,Alice,IT,81119.0
174,Diana,IT,83696.0


In [37]:
# query() — SQL-подібний синтаксис
filtered_q = df_f.query('department == "IT" and salary > 60000 and is_active == True')
print(f'query() result: {len(filtered_q)}')

# isin() — фільтр за списком
filtered_isin = df_f[df_f['department'].isin(['IT', 'Finance'])]
print(f'isin(): {len(filtered_isin)}')

# between() — діапазон
filtered_age = df_f[df_f['age'].between(25, 35)]
print(f'between(25,35): {len(filtered_age)}')

query() result: 29
isin(): 417
between(25,35): 251


In [38]:
# Сортування
print('=== Топ-5 за зарплатою ===')
display(df_f.sort_values('salary', ascending=False)[['name', 'salary', 'department']].head(5))

print('\n=== Сортування за двома колонками ===')
display(df_f.sort_values(['department', 'salary'], ascending=[True, False]).head(5))

=== Топ-5 за зарплатою ===


,name,salary,department
323,Bob,179253.0,Finance
665,Alice,174359.0,IT
743,Charlie,144326.0,Marketing
642,Bob,136240.0,Finance
161,Diana,133365.0,Finance



=== Сортування за двома колонками ===


,user_id,name,age,salary,department,join_date,is_active,score
323,324,Bob,28.0,179253.0,Finance,2020-12-09,True,81.1
642,643,Bob,55.0,136240.0,Finance,2021-01-18,False,84.4
161,162,Diana,30.0,133365.0,Finance,2019-07-26,True,97.7
828,829,Diana,39.0,121335.0,Finance,2019-11-13,True,81.1
619,620,Diana,43.0,110819.0,Finance,2021-03-25,True,55.0


---
## 7. GroupBy та Агрегація

Хліб DE — групування і зведення даних.

In [39]:
# Базова агрегація
print('=== Середня зарплата по департаментах ===')
display(df_f.groupby('department')['salary'].agg(['mean', 'median', 'std', 'min', 'max', 'count']).round(0))

=== Середня зарплата по департаментах ===


,mean,median,std,min,max,count
department,,,,,,
Finance,42338.0,35837.0,24000.0,10128.0,179253.0,209
HR,40923.0,37420.0,19249.0,8024.0,107524.0,185
IT,43136.0,36695.0,23138.0,8348.0,174359.0,203
Marketing,40950.0,35542.0,21452.0,9876.0,144326.0,182
Sales,39519.0,36150.0,18247.0,9942.0,99888.0,189


In [40]:
# Множинні агрегації для різних колонок
agg_result = df_f.groupby('department').agg({
    'salary': ['mean', 'std', 'count'],
    'age': 'mean',
    'score': 'median',
    'is_active': 'sum',
    'user_id': 'nunique'
})
display(agg_result.round(1))

salary                  age  score is_active user_id
               mean      std count  mean median       sum nunique
department                                                       
Finance     42338.2  23999.9   209  42.1   55.4       183     210
HR          40922.6  19248.9   185  41.2   45.6       156     190
IT          43136.3  23138.4   203  40.5   49.2       177     207
Marketing   40949.8  21451.5   182  39.5   45.9       158     185
Sales       39518.9  18247.4   189  41.5   49.2       162     193

In [41]:
# Named aggregation (cleaner syntax, pandas 0.25+)
agg_named = df_f.groupby('department', as_index=False).agg(
    avg_salary=('salary', 'mean'),
    max_salary=('salary', 'max'),
    avg_score=('score', 'mean'),
    employee_count=('user_id', 'nunique'),
    active_count=('is_active', 'sum')
)
display(agg_named.round(1))

,department,avg_salary,max_salary,avg_score,employee_count,active_count
0,Finance,42338.2,179253.0,52.4,210,183
1,HR,40922.6,107524.0,48.3,190,156
2,IT,43136.3,174359.0,51.3,207,177
3,Marketing,40949.8,144326.0,45.7,185,158
4,Sales,39518.9,99888.0,48.8,193,162


In [42]:
# transform — застосувати агрегацію до кожного рядка
df_f['dept_avg_salary'] = df_f.groupby('department')['salary'].transform('mean')
df_f['salary_ratio'] = (df_f['salary'] / df_f['dept_avg_salary']).round(2)
display(df_f[['name', 'department', 'salary', 'dept_avg_salary', 'salary_ratio']].head(10))

,name,department,salary,dept_avg_salary,salary_ratio
0,Bob,NaN,NaN,NaN,NaN
1,NaN,HR,55229.0,40922.551351,1.35
2,Diana,HR,37832.0,40922.551351,0.92
3,Charlie,Finance,34564.0,42338.200957,0.82
4,Alice,HR,57500.0,40922.551351,1.41
5,Alice,IT,31409.0,43136.251232,0.73
6,Alice,IT,41510.0,43136.251232,0.96
7,Eve,Finance,42653.0,42338.200957,1.01
8,Diana,HR,26003.0,40922.551351,0.64
9,Diana,IT,59636.0,43136.251232,1.38


In [43]:
# Pivot table — зведені таблиці
pivot = pd.pivot_table(
    df_f,
    values='salary',
    index='department',
    columns='is_active',
    aggfunc='mean',
    margins=True
).round(0)
display(pivot)

is_active,False,True,All
department,,,
Finance,42626.0,42296.0,42338.0
HR,43024.0,40483.0,40923.0
IT,41662.0,43392.0,43136.0
Marketing,41438.0,40865.0,40950.0
Sales,42174.0,39018.0,39519.0
All,42203.0,41285.0,41423.0


In [44]:
# Crosstab — для категоріальних даних
ct = pd.crosstab(df_f['department'], df_f['is_active'], margins=True, normalize='index')
display(ct.round(2))

is_active,False,True
department,,
Finance,0.13,0.87
HR,0.18,0.82
IT,0.14,0.86
Marketing,0.15,0.85
Sales,0.16,0.84
All,0.15,0.85


In [45]:
# Ранжування всередині груп
df_f['salary_rank_in_dept'] = df_f.groupby('department')['salary'].rank(ascending=False)
display(df_f[df_f['salary_rank_in_dept'] <= 3][['name', 'department', 'salary', 'salary_rank_in_dept']].sort_values(['department', 'salary_rank_in_dept']))

,name,department,salary,salary_rank_in_dept
323,Bob,Finance,179253.0,1.0
642,Bob,Finance,136240.0,2.0
161,Diana,Finance,133365.0,3.0
474,Charlie,HR,107524.0,1.0
632,Charlie,HR,102441.0,2.0
134,NaN,HR,101529.0,3.0
665,Alice,IT,174359.0,1.0
199,Eve,IT,130496.0,2.0
926,Bob,IT,103763.0,3.0
743,Charlie,Marketing,144326.0,1.0


---
## 8. Об'єднання даних: Merge, Concat, Join

В DE ми постійно об'єднуємо дані з різних джерел.

In [46]:
# Демо-дані для join
employees = pd.DataFrame({
    'emp_id': [1, 2, 3, 4, 5],
    'name': ['Alice', 'Bob', 'Charlie', 'Diana', 'Eve'],
    'dept_id': [10, 20, 10, 30, 20]
})

departments = pd.DataFrame({
    'dept_id': [10, 20, 30, 40],
    'dept_name': ['IT', 'HR', 'Finance', 'R&D'],
    'budget': [500000, 300000, 400000, 600000]
})

salaries = pd.DataFrame({
    'emp_id': [1, 2, 3, 4, 5, 6],
    'salary': [70000, 50000, 80000, 55000, 65000, 90000],
    'month': ['2024-01', '2024-01', '2024-01', '2024-01', '2024-01', '2024-01']
})

print('employees:'); display(employees)
print('departments:'); display(departments)
print('salaries:'); display(salaries)

employees:


,emp_id,name,dept_id
0,1,Alice,10
1,2,Bob,20
2,3,Charlie,10
3,4,Diana,30
4,5,Eve,20


departments:


,dept_id,dept_name,budget
0,10,IT,500000
1,20,HR,300000
2,30,Finance,400000
3,40,R&D,600000


salaries:


,emp_id,salary,month
0,1,70000,2024-01
1,2,50000,2024-01
2,3,80000,2024-01
3,4,55000,2024-01
4,5,65000,2024-01
5,6,90000,2024-01


In [47]:
# INNER JOIN (default)
inner = pd.merge(employees, departments, on='dept_id', how='inner')
print('INNER JOIN:'); display(inner)

INNER JOIN:


,emp_id,name,dept_id,dept_name,budget
0,1,Alice,10,IT,500000
1,2,Bob,20,HR,300000
2,3,Charlie,10,IT,500000
3,4,Diana,30,Finance,400000
4,5,Eve,20,HR,300000


In [48]:
# LEFT / RIGHT / OUTER JOIN
left = pd.merge(employees, departments, on='dept_id', how='left')
right = pd.merge(employees, departments, on='dept_id', how='right')
outer = pd.merge(employees, departments, on='dept_id', how='outer')

print('LEFT JOIN:'); display(left)
print('RIGHT JOIN:'); display(right)
print('FULL OUTER JOIN:'); display(outer)

LEFT JOIN:


,emp_id,name,dept_id,dept_name,budget
0,1,Alice,10,IT,500000
1,2,Bob,20,HR,300000
2,3,Charlie,10,IT,500000
3,4,Diana,30,Finance,400000
4,5,Eve,20,HR,300000


RIGHT JOIN:


,emp_id,name,dept_id,dept_name,budget
0,1.0,Alice,10,IT,500000
1,3.0,Charlie,10,IT,500000
2,2.0,Bob,20,HR,300000
3,5.0,Eve,20,HR,300000
4,4.0,Diana,30,Finance,400000
5,NaN,NaN,40,R&D,600000


FULL OUTER JOIN:


,emp_id,name,dept_id,dept_name,budget
0,1.0,Alice,10,IT,500000
1,3.0,Charlie,10,IT,500000
2,2.0,Bob,20,HR,300000
3,5.0,Eve,20,HR,300000
4,4.0,Diana,30,Finance,400000
5,NaN,NaN,40,R&D,600000


In [49]:
# Об'єднання за різними ключами
merged = pd.merge(employees, salaries, on='emp_id', how='inner')
merged2 = pd.merge(merged, departments, on='dept_id', how='left')
print('3-way merge:'); display(merged2)

# Перейменування колонок при конфлікті
merged_suffix = pd.merge(employees, departments.rename(columns={'dept_id': 'department_id'}),
                         left_on='dept_id', right_on='department_id', how='left')
print('Merge з різними ключами:'); display(merged_suffix)

3-way merge:


,emp_id,name,dept_id,salary,month,dept_name,budget
0,1,Alice,10,70000,2024-01,IT,500000
1,2,Bob,20,50000,2024-01,HR,300000
2,3,Charlie,10,80000,2024-01,IT,500000
3,4,Diana,30,55000,2024-01,Finance,400000
4,5,Eve,20,65000,2024-01,HR,300000


Merge з різними ключами:


,emp_id,name,dept_id,department_id,dept_name,budget
0,1,Alice,10,10,IT,500000
1,2,Bob,20,20,HR,300000
2,3,Charlie,10,10,IT,500000
3,4,Diana,30,30,Finance,400000
4,5,Eve,20,20,HR,300000


In [50]:
# Concat — склеювання рядків/стовпців
q1 = pd.DataFrame({'emp_id': [1, 2], 'revenue': [100, 200], 'quarter': 'Q1'})
q2 = pd.DataFrame({'emp_id': [1, 2], 'revenue': [150, 250], 'quarter': 'Q2'})
q3 = pd.DataFrame({'emp_id': [1, 2], 'revenue': [130, 220], 'quarter': 'Q3'})

# Вертикальне об'єднання
all_quarters = pd.concat([q1, q2, q3], ignore_index=True)
print('Concat (вертикально):')
display(all_quarters)

# Горизонтальне об'єднання
wide = pd.concat([q1.set_index('emp_id'), q2.set_index('emp_id')[['revenue']].rename(columns={'revenue': 'revenue_q2'}),
                  q3.set_index('emp_id')[['revenue']].rename(columns={'revenue': 'revenue_q3'})], axis=1)
print('Concat (горизонтально):')
display(wide)

Concat (вертикально):


,emp_id,revenue,quarter
0,1,100,Q1
1,2,200,Q1
2,1,150,Q2
3,2,250,Q2
4,1,130,Q3
5,2,220,Q3


Concat (горизонтально):


,revenue,quarter,revenue_q2,revenue_q3
emp_id,,,,
1,100,Q1,150,130
2,200,Q1,250,220


In [51]:
# Join (на індексі)
emp_idx = employees.set_index('emp_id')
sal_idx = salaries.groupby('emp_id')['salary'].mean().to_frame('avg_salary')
joined = emp_idx.join(sal_idx, how='left')
print('Join на індексі:'); display(joined)

Join на індексі:


,name,dept_id,avg_salary
emp_id,,,
1,Alice,10,70000.0
2,Bob,20,50000.0
3,Charlie,10,80000.0
4,Diana,30,55000.0
5,Eve,20,65000.0


---
## 9. Робота з датами та часом

Для DE це одна з найчастіших операцій.

In [54]:
# 1. Створення датафрейму з різними форматами дат
dates = pd.DataFrame({
    'date_str': ['2024-01-15', '2024/02/20', '15-03-2024', '2024.04.10', 'May 5, 2024'],
    'value': [1, 2, 3, 4, 5]
})

# 2. Парсинг із параметром format='mixed'
# Він змусить Pandas аналізувати кожен рядок окремо
dates['parsed'] = pd.to_datetime(dates['date_str'], format='mixed')

print('Парсинг різних форматів:')
display(dates)

print('\nDtypes:')
print(dates.dtypes)

Парсинг різних форматів:


,date_str,value,parsed
0,2024-01-15,1,2024-01-15
1,2024/02/20,2,2024-02-20
2,15-03-2024,3,2024-03-15
3,2024.04.10,4,2024-04-10
4,"May 5, 2024",5,2024-05-05



Dtypes:
date_str               str
value                int64
parsed      datetime64[us]
dtype: object


In [ ]:
# Виділення компонентів дати
df_dates = df_f.copy()
df_dates['join_date'] = pd.to_datetime(df_dates['join_date'])

df_dates['year'] = df_dates['join_date'].dt.year
df_dates['month'] = df_dates['join_date'].dt.month
df_dates['day'] = df_dates['join_date'].dt.day
df_dates['dayofweek'] = df_dates['join_date'].dt.dayofweek  # 0=Monday
df_dates['quarter'] = df_dates['join_date'].dt.quarter
df_dates['week'] = df_dates['join_date'].dt.isocalendar().week.astype(int)
df_dates['is_weekend'] = df_dates['join_date'].dt.dayofweek >= 5

display(df_dates[['join_date', 'year', 'month', 'quarter', 'dayofweek', 'is_weekend']].head(10))

,join_date,year,month,quarter,dayofweek,is_weekend
0,2019-11-01,2019,11,4,4,False
1,2018-05-29,2018,5,2,1,False
2,2020-09-11,2020,9,3,4,False
3,2020-05-01,2020,5,2,4,False
4,2019-09-20,2019,9,3,4,False
5,2019-03-22,2019,3,1,4,False
6,2020-06-02,2020,6,2,1,False
7,2018-10-21,2018,10,4,6,True
8,2021-03-12,2021,3,1,4,False
9,2018-09-07,2018,9,3,4,False


In [53]:
# Date range та ресемплінг (для часових рядів)
date_range = pd.date_range(start='2024-01-01', end='2024-12-31', freq='D')
ts_df = pd.DataFrame({
    'date': date_range,
    'sales': np.random.randint(100, 500, size=len(date_range))
})

# Ресемплінг: денний -> місячний
ts_monthly = ts_df.set_index('date').resample('ME').agg({'sales': ['sum', 'mean', 'count']})
print('Місячна агрегація:'); display(ts_monthly.round(0))

# Змінна ширина вікна (rolling)
ts_df['rolling_7d'] = ts_df.set_index('date')['sales'].rolling(window=7).mean().values
display(ts_df.head(15))

Місячна агрегація:


sales             
             sum   mean count
date                         
2024-01-31  9301  300.0    31
2024-02-29  9685  334.0    29
2024-03-31  9724  314.0    31
2024-04-30  9008  300.0    30
2024-05-31  9386  303.0    31
2024-06-30  8069  269.0    30
2024-07-31  9405  303.0    31
2024-08-31  8233  266.0    31
2024-09-30  9280  309.0    30
2024-10-31  9465  305.0    31
2024-11-30  8991  300.0    30
2024-12-31  9696  313.0    31

,date,sales,rolling_7d
0,2024-01-01,227,NaN
1,2024-01-02,386,NaN
2,2024-01-03,410,NaN
3,2024-01-04,352,NaN
4,2024-01-05,365,NaN
5,2024-01-06,287,NaN
6,2024-01-07,485,358.857143
7,2024-01-08,329,373.428571
8,2024-01-09,370,371.142857
9,2024-01-10,358,363.714286


In [ ]:
# Різниця в датах
df_dates['days_since_join'] = (datetime.now() - df_dates['join_date']).dt.days
df_dates['years_since_join'] = (df_dates['days_since_join'] / 365.25).round(1)

display(df_dates[['name', 'join_date', 'days_since_join', 'years_since_join']].head(10))

,name,join_date,days_since_join,years_since_join
0,Bob,2019-11-01,2396,6.6
1,NaN,2018-05-29,2917,8.0
2,Diana,2020-09-11,2081,5.7
3,Charlie,2020-05-01,2214,6.1
4,Alice,2019-09-20,2438,6.7
5,Alice,2019-03-22,2620,7.2
6,Alice,2020-06-02,2182,6.0
7,Eve,2018-10-21,2772,7.6
8,Diana,2021-03-12,1899,5.2
9,Diana,2018-09-07,2816,7.7


---
## 10. Розширені операції

Apply, Lambda, Window Functions, String Operations.

In [ ]:
# apply() — увага: повільний! Використовувати тільки якщо немає vectorized solution
df_apply = df_f.head(100).copy()
df_apply['salary_category'] = df_apply['salary'].apply(
    lambda x: 'high' if x > 70000 else ('medium' if x > 40000 else 'low')
)
display(df_apply[['name', 'salary', 'salary_category']].head(10))

,name,salary,salary_category
0,Bob,NaN,low
1,NaN,55229.0,medium
2,Diana,37832.0,low
3,Charlie,34564.0,low
4,Alice,57500.0,medium
5,Alice,31409.0,low
6,Alice,41510.0,medium
7,Eve,42653.0,medium
8,Diana,26003.0,low
9,Diana,59636.0,medium


In [ ]:
# String методи (векторизовані!)
df_str = pd.DataFrame({
    'email': ['alice@company.com', 'BOB@COMPANY.COM', 'Charlie@Company.Com', None],
    'phone': ['+380-50-123-45-67', '380631234567', '+38(067)123-45-67', None]
})

df_str['email_lower'] = df_str['email'].str.lower()
df_str['email_domain'] = df_str['email'].str.split('@').str[1]
df_str['has_company_email'] = df_str['email'].str.contains('company', case=False, na=False)
df_str['phone_clean'] = df_str['phone'].str.replace(r'[\s\-\(\)\+]', '', regex=True)

display(df_str)

,email,phone,email_lower,email_domain,has_company_email,phone_clean
0,alice@company.com,+380-50-123-45-67,alice@company.com,company.com,True,380501234567
1,BOB@COMPANY.COM,380631234567,bob@company.com,COMPANY.COM,True,380631234567
2,Charlie@Company.Com,+38(067)123-45-67,charlie@company.com,Company.Com,True,380671234567
3,NaN,NaN,NaN,NaN,False,NaN


In [ ]:
# Window Functions (аналог SQL-віконних функцій)
df_window = df_f.sort_values(['department', 'salary']).copy()

# row_number = rank без пропусків
df_window['rn'] = df_window.groupby('department')['salary'].cumcount() + 1

# rank
df_window['rank_salary'] = df_window.groupby('department')['salary'].rank(method='dense', ascending=False)

# lag / lead через shift
df_window['prev_salary'] = df_window.groupby('department')['salary'].shift(1)
df_window['salary_diff'] = df_window['salary'] - df_window['prev_salary']

# cumulative sum
df_window['cumsum_salary'] = df_window.groupby('department')['salary'].cumsum()

display(df_window[df_window['rank_salary'] <= 3][['department', 'name', 'salary', 'rank_salary', 'prev_salary', 'salary_diff']].head(10))

,department,name,salary,rank_salary,prev_salary,salary_diff
161,Finance,Diana,133365.0,3.0,121335.0,12030.0
642,Finance,Bob,136240.0,2.0,133365.0,2875.0
323,Finance,Bob,179253.0,1.0,136240.0,43013.0
134,HR,NaN,101529.0,3.0,85464.0,16065.0
632,HR,Charlie,102441.0,2.0,101529.0,912.0
474,HR,Charlie,107524.0,1.0,102441.0,5083.0
926,IT,Bob,103763.0,3.0,101851.0,1912.0
199,IT,Eve,130496.0,2.0,103763.0,26733.0
665,IT,Alice,174359.0,1.0,130496.0,43863.0
883,Marketing,Alice,103236.0,3.0,101595.0,1641.0


In [ ]:
# map та replace
dept_map = {'IT': 'Technology', 'HR': 'Human Resources', 'Finance': 'Financial', 'Marketing': 'Marketing', 'Sales': 'Sales'}
df_f['dept_full'] = df_f['department'].map(dept_map)

# replace — для заміни значень
df_f['is_active_label'] = df_f['is_active'].replace({True: 'Active', False: 'Inactive'})

display(df_f[['department', 'dept_full', 'is_active', 'is_active_label']].head(10))

,department,dept_full,is_active,is_active_label
0,NaN,NaN,True,Active
1,HR,Human Resources,True,Active
2,HR,Human Resources,True,Active
3,Finance,Financial,True,Active
4,HR,Human Resources,False,Inactive
5,IT,Technology,True,Active
6,IT,Technology,False,Inactive
7,Finance,Financial,True,Active
8,HR,Human Resources,True,Active
9,IT,Technology,True,Active


---
## 11. Polars — сучасна альтернатива Pandas

**Polars** — бібліотека для роботи з DataFrame, написана на Rust.

| Характеристика | Pandas | Polars |
|---|---|---|
| Мова ядра | Python / C / Cython | Rust |
| Ліниві обчислення | Ні | Так (LazyFrame) |
| Багатонитковість | Обмежена | Нативна (всі ядра) |
| Використання пам'яті | Висока | Ефективна (нульова копія) |
| Вирази (Expression API) | Обмежено | Повноцінний |
| Індекс | Так (важлива частина) | Немає (проста таблиця) |
| Екосистема | Величезна | Зростаюча |

> **Коли Pandas:** потрібна багата екосистема, ML, візуалізація, робота з малими/середніми даними (< RAM).
> **Коли Polars:** великі дані (10-100 ГБ), продуктивність критична, багатонитковість, строга типізація.

In [ ]:
# Якщо Polars встановлено — покажемо порівняння
try:
    import polars as pl
    print(f'Polars version: {pl.__version__}')
    
    # Створення DataFrame
    df_pl = pl.DataFrame({
        'user_id': [1, 2, 3, 4, 5],
        'name': ['Alice', 'Bob', 'Charlie', 'Diana', 'Eve'],
        'salary': [70000, 50000, 80000, 55000, 65000],
        'department': ['IT', 'HR', 'IT', 'Finance', 'HR']
    })
    print('\nPolars DataFrame:')
    print(df_pl)
    
except ImportError:
    print('Polars не встановлено. Встановіть: pip install polars')
    print('\nПриклад коду нижче (для довідки):')

Polars не встановлено. Встановіть: pip install polars

Приклад коду нижче (для довідки):


In [ ]:
# Polars: Expression API (лінивий режим)
try:
    import polars as pl
    
    # LazyFrame — будуємо план виконання
    q = (
        pl.read_csv(f'{demo_dir}/employees.csv')
        .lazy()
        .filter(pl.col('salary') > 55000)
        .group_by('department')
        .agg([
            pl.col('salary').mean().alias('avg_salary'),
            pl.col('salary').max().alias('max_salary'),
            pl.col('user_id').count().alias('emp_count')
        ])
        .sort('avg_salary', descending=True)
    )
    
    print('=== Polars LazyFrame план ===')
    print(q.explain())
    print('\n=== Результат ===')
    print(q.collect())
    
    # Порівняння продуктивності
    import time
    
    # Pandas
    start = time.time()
    for _ in range(10):
        pdf = pd.read_csv(f'{demo_dir}/employees.csv')
        pdf[pdf['salary'] > 55000].groupby('department')['salary'].agg(['mean', 'max', 'count'])
    pd_time = time.time() - start
    
    # Polars
    start = time.time()
    for _ in range(10):
        pdf_pl = pl.read_csv(f'{demo_dir}/employees.csv')
        pdf_pl.filter(pl.col('salary') > 55000).group_by('department').agg([
            pl.col('salary').mean().alias('avg_salary'),
            pl.col('salary').max().alias('max_salary'),
            pl.col('user_id').count().alias('emp_count')
        ])
    pl_time = time.time() - start
    
    print(f'\n=== Performance: 10 iterations ===')
    print(f'Pandas: {pd_time:.3f} sec')
    print(f'Polars: {pl_time:.3f} sec')
    print(f'Polars швидше в {(pd_time / pl_time):.1f}x разів')
    
except ImportError:
    print('Для запуску Polars-коду: pip install polars')

=== Polars LazyFrame план ===
SORT BY [descending: [true]] [col("avg_salary")]
  AGGREGATE[maintain_order: false]
    [col("salary").mean().alias("avg_salary"), col("salary").max().alias("max_salary"), col("user_id").count().alias("emp_count")] BY [col("department")]
    FROM
    FILTER [(col("salary")) > (55000)]
    FROM
      DF ["user_id", "name", "age", "salary", ...]; PROJECT["department", "salary", "user_id"] 3/5 COLUMNS

=== Результат ===
shape: (2, 4)
┌────────────┬────────────┬────────────┬───────────┐
│ department ┆ avg_salary ┆ max_salary ┆ emp_count │
│ ---        ┆ ---        ┆ ---        ┆ ---       │
│ str        ┆ f64        ┆ i64        ┆ u32       │
╞════════════╪════════════╪════════════╪═══════════╡
│ HR         ┆ 70000.0    ┆ 80000      ┆ 2         │
│ IT         ┆ 70000.0    ┆ 70000      ┆ 1         │
└────────────┴────────────┴────────────┴───────────┘

=== Performance: 10 iterations ===
Pandas: 0.018 sec
Polars: 0.006 sec
Polars швидше в 3.2x разів


In [ ]:
# Polars — ключові відмінності в синтаксисі
try:
    import polars as pl
    
    # Polars не має index — це просто таблиця
    # Замість df.iloc / df.loc — використовуємо filter + expressions
    
    df_pl = pl.DataFrame({
        'name': ['Alice', 'Bob', 'Charlie', 'Diana'],
        'age': [25, 30, 35, 28],
        'salary': [70000, 50000, 80000, 55000],
        'dept': ['IT', 'HR', 'IT', 'Finance']
    })
    
    # Pandas style: df[df['age'] > 28]
    # Polars style:
    result = df_pl.filter(pl.col('age') > 28)
    print('Filter:'); print(result)
    
    # З SQL-подібним синтаксисом:
    result2 = df_pl.sql('SELECT name, salary FROM self WHERE age > 28 ORDER BY salary DESC')
    print('\nSQL-like:'); print(result2)
    
except ImportError:
    print('Polars not installed')

Filter:
shape: (2, 4)
┌─────────┬─────┬────────┬──────┐
│ name    ┆ age ┆ salary ┆ dept │
│ ---     ┆ --- ┆ ---    ┆ ---  │
│ str     ┆ i64 ┆ i64    ┆ str  │
╞═════════╪═════╪════════╪══════╡
│ Bob     ┆ 30  ┆ 50000  ┆ HR   │
│ Charlie ┆ 35  ┆ 80000  ┆ IT   │
└─────────┴─────┴────────┴──────┘

SQL-like:
shape: (2, 2)
┌─────────┬────────┐
│ name    ┆ salary │
│ ---     ┆ ---    │
│ str     ┆ i64    │
╞═════════╪════════╡
│ Charlie ┆ 80000  │
│ Bob     ┆ 50000  │
└─────────┴────────┘


---
## 12. Performance Tips для Data Engineering

Поради для ефективної роботи з Pandas в продакшн-пайплайнах.

In [ ]:
# 1. Використовуйте правильні типи даних
print('=== Memory optimization ===')

# Замість float64 -> float32
df_opt = pd.DataFrame({'value': np.random.randn(100000)})
print(f'float64: {df_opt["value"].memory_usage(deep=True) / 1024:.1f} KB')

df_opt['value'] = df_opt['value'].astype('float32')
print(f'float32: {df_opt["value"].memory_usage(deep=True) / 1024:.1f} KB')

# int64 -> int8/16/32 (якщо діапазон дозволяє)
df_opt['small_int'] = np.random.randint(0, 100, 100000)
print(f'\nint64: {df_opt["small_int"].memory_usage(deep=True) / 1024:.1f} KB')
df_opt['small_int'] = df_opt['small_int'].astype('int8')
print(f'int8: {df_opt["small_int"].memory_usage(deep=True) / 1024:.1f} KB')

=== Memory optimization ===
float64: 781.4 KB
float32: 390.8 KB

int64: 390.8 KB
int8: 97.8 KB


In [ ]:
# 2. category type для категоріальних даних
df_cat = pd.DataFrame({'dept': np.random.choice(['IT', 'HR', 'Finance', 'Marketing', 'Sales'], size=100000)})
print(f'object: {df_cat["dept"].memory_usage(deep=True) / 1024:.1f} KB')

df_cat['dept'] = df_cat['dept'].astype('category')
print(f'category: {df_cat["dept"].memory_usage(deep=True) / 1024:.1f} KB')

object: 1270.4 KB
category: 97.8 KB


In [ ]:
# 3. Уникайте apply() — використовуйте векторизовані операції
n = 100000
df_perf = pd.DataFrame({'a': np.random.randn(n), 'b': np.random.randn(n)})

import time

# Повільно (apply)
start = time.time()
df_perf['c_apply'] = df_perf.apply(lambda row: row['a'] + row['b'], axis=1)
t_apply = time.time() - start

# Швидко (vectorized)
start = time.time()
df_perf['c_vec'] = df_perf['a'] + df_perf['b']
t_vec = time.time() - start

print(f'apply(): {t_apply:.4f} sec')
print(f'vectorized: {t_vec:.4f} sec')
print(f'Векторизація швидше в {t_apply / t_vec:.0f}x разів')

apply(): 0.3520 sec
vectorized: 0.0010 sec
Векторизація швидше в 353x разів


In [ ]:
# 4. inplace=True vs копія
# inplace=True не завжди швидше і може бути неінтуїтивним
# Краще: df = df.drop(...)  # явне присвоєння
# Ніж: df.drop(..., inplace=True)  # прихована модифікація

# 5. Читання великих файлів
# - usecols: читати тільки потрібні колонки
# - dtype: задати типи при читанні (економить пам'ять)
# - nrows: спочатку прочитати невелику вибірку
# - chunksize: читати частинами

print('Читання з оптимізаціями:')
# pd.read_csv('file.csv', usecols=['id', 'name', 'salary'], dtype={'id': 'int32', 'salary': 'float32'}, nrows=1000)

# 6. Використовуйте query() замість boolean indexing (трохи швидше)
# 7. Зберігайте в Parquet замість CSV (швидше читання/запис, менше місця)

Читання з оптимізаціями:


In [ ]:
# ETL-шаблон: типовий пайплайн DE з Pandas
def etl_pipeline(filepath: str) -> pd.DataFrame:
    """
    Типовий ETL-пайплайн:
    - Extract: читання CSV
    - Transform: очищення + трансформація
    - Load: готовий DataFrame
    """
    # Extract
    df = pd.read_csv(filepath)
    
    # Transform
    # 1. Типи
    df['join_date'] = pd.to_datetime(df['join_date'], errors='coerce')
    
    # 2. Пропуски
    df['salary'] = df.groupby('department')['salary'].transform(lambda x: x.fillna(x.median()))
    df['department'] = df['department'].fillna('Unknown')
    
    # 3. Фільтрація
    df = df[df['is_active'] == True]
    
    # 4. Нові колонки
    df['years_exp'] = (pd.Timestamp.now() - df['join_date']).dt.days / 365.25
    
    # 5. Агрегація (для звіту)
    report = df.groupby('department', as_index=False).agg(
        emp_count=('user_id', 'count'),
        avg_salary=('salary', 'mean'),
        avg_years=('years_exp', 'mean')
    ).round(1)
    
    return report

print('ETL-функція визначена. Запустіть:')
print('result = etl_pipeline("demo_data/employees.csv")')

ETL-функція визначена. Запустіть:
result = etl_pipeline("demo_data/employees.csv")


---
## 13. Практичні завдання (Self-Check)

Спробуйте виконати самостійно:

1. **Завантажте** `demo_data/employees.csv`
2. **Знайдіть** топ-3 відділи за середньою зарплатою
3. **Створіть** колонку `age_group`: young (<30), mid (30-45), senior (>45)
4. **Зробіть** звіт: середня зарплата по `department` + `age_group`
5. **Збережіть** результат у Parquet
6. **Переробіть** той самий пайплайн на Polars (якщо встановлено)

In [ ]:
# === Розв'язання практичного завдання ===

# 0. Завантаження даних
df_practice = pd.read_csv(f'{demo_dir}/employees.csv')

# 1. Топ-3 відділи за середньою зарплатою
top_depts = df_practice.groupby('department')['salary'].mean().sort_values(ascending=False).head(3)
print('Топ-3 відділи за середньою зарплатою:')
display(top_depts.round(0))

# 2. Вікова група
df_practice['age_group'] = pd.cut(df_practice['age'], 
    bins=[0, 30, 45, 200],
    labels=['young', 'mid', 'senior']
)

# 3. Звіт
report = df_practice.pivot_table(
    values='salary',
    index='department',
    columns='age_group',
    aggfunc='mean'
).round(0)

# ВИПРАВЛЕННЯ ПОМИЛКИ: Перетворюємо категоріальні назви колонок на звичайні рядки,
# щоб бібліотека PyArrow (або fastparquet) могла без проблем серіалізувати їх у Parquet.
report.columns = report.columns.astype(str)

print('\nЗвіт: середня зарплата по department + age_group:')
display(report)

# 4. Збереження
report.to_parquet(f'{demo_dir}/salary_report.parquet', engine='pyarrow') 
print('\nЗбережено в demo_data/salary_report.parquet')

Топ-3 відділи за середньою зарплатою:


department
HR         70000.0
IT         60000.0
Finance    55000.0
Name: salary, dtype: float64


Звіт: середня зарплата по department + age_group:


age_group,young,mid
department,,
Finance,55000.0,NaN
HR,60000.0,80000.0
IT,50000.0,70000.0



Збережено в demo_data/salary_report.parquet


---
## Висновки

- **Pandas** — must-have для Data Engineer: читання, очищення, трансформація, агрегація
- **Головне правило**: векторизовані операції > apply() > loops
- **Оптимізація**: правильні типи, category, Parquet, chunked reading
- **Polars** — наступний крок: якщо дані великі або продуктивність критична

> 💡 **Порада DE**: спочатку завжди досліджуйте дані (info, describe, isnull), потім пишіть трансформацію.

In [ ]:
# Додаткові матеріали
print('Корисні посилання:')
print('• Pandas docs: https://pandas.pydata.org/docs/')
print('• Pandas Cheat Sheet: https://pandas.pydata.org/Pandas_Cheat_Sheet.pdf')
print('• Polars docs: https://docs.pola.rs/')
print('• Python Data Science Handbook: https://jakevdp.github.io/PythonDataScienceHandbook/')

Корисні посилання:
• Pandas docs: https://pandas.pydata.org/docs/
• Pandas Cheat Sheet: https://pandas.pydata.org/Pandas_Cheat_Sheet.pdf
• Polars docs: https://docs.pola.rs/
• Python Data Science Handbook: https://jakevdp.github.io/PythonDataScienceHandbook/
